In [ ]:
import pandas as pd
import numpy as np
import json

INPUT_DIR = "<link-to-the-1st-notebook>"

subset = pd.read_parquet(f"{INPUT_DIR}/retailcast_subset.parquet")
with open(f"{INPUT_DIR}/subset_config.json") as f:
    config = json.load(f)

subset["date"] = pd.to_datetime(subset["date"])
subset = subset.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)
print(subset.shape)

In [ ]:
activation_diag = pd.read_csv(
    f"{INPUT_DIR}/series_activation_diagnostics.csv", parse_dates=["activation_date"]
)
subset = subset.merge(
    activation_diag[["store_nbr", "family", "activation_date"]],
    on=["store_nbr", "family"], how="left",
)
before = len(subset)
subset = subset[subset["date"] >= subset["activation_date"]].drop(columns=["activation_date"])
print(f"Trimmed {before - len(subset)} pre-activation rows across all series")

In [ ]:
def classify_demand_pattern(series):
    nonzero = series[series > 0]
    n_periods = len(series)
    n_nonzero = len(nonzero)
    if n_nonzero == 0:
        return np.nan, np.nan, "no_demand"

    adi = n_periods / n_nonzero
    cv2 = (nonzero.std() / nonzero.mean()) ** 2 if nonzero.mean() > 0 else np.nan

    if adi < 1.32 and cv2 < 0.49:
        label = "smooth"
    elif adi >= 1.32 and cv2 < 0.49:
        label = "intermittent"
    elif adi < 1.32 and cv2 >= 0.49:
        label = "erratic"
    else:
        label = "lumpy"
    return adi, cv2, label

pattern_results = []
for (store_nbr, family), g in subset.groupby(["store_nbr", "family"]):
    s = g.sort_values("date")["sales"]
    adi, cv2, label = classify_demand_pattern(s)
    pattern_results.append({"store_nbr": store_nbr, "family": family, "adi": adi, "cv2": cv2, "pattern": label})

pattern_df = pd.DataFrame(pattern_results)
print(pattern_df.sort_values("pattern"))
print("\nPattern counts:\n", pattern_df["pattern"].value_counts())
pattern_df.to_csv("/kaggle/working/demand_pattern_classification.csv", index=False)

In [ ]:
subset["day_of_week"] = subset["date"].dt.dayofweek
subset["month"] = subset["date"].dt.month
subset["is_weekend"] = (subset["day_of_week"] >= 5).astype(int)
subset["day_of_month"] = subset["date"].dt.day
subset["week_of_year"] = subset["date"].dt.isocalendar().week.astype(int)

In [ ]:
def add_lag_rolling_features(df):
    df = df.copy()
    group_cols = ["store_nbr", "family"]

    for lag in [7, 14, 28]:
        df[f"lag_{lag}"] = df.groupby(group_cols)["sales"].shift(lag)

    for window in [7, 14, 28]:
        shifted = df.groupby(group_cols)["sales"].shift(1)  # avoid leakage: exclude current day
        df[f"rolling_mean_{window}"] = (
            shifted.groupby([df["store_nbr"], df["family"]])
            .rolling(window, min_periods=max(2, window // 2))
            .mean()
            .reset_index(level=[0, 1], drop=True)
        )
        df[f"rolling_std_{window}"] = (
            shifted.groupby([df["store_nbr"], df["family"]])
            .rolling(window, min_periods=max(2, window // 2))
            .std()
            .reset_index(level=[0, 1], drop=True)
        )
    return df

subset = add_lag_rolling_features(subset)
print(subset[["lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_std_7"]].isna().sum())

In [ ]:
categorical_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]
for col in categorical_cols:
    subset[col] = subset[col].astype("category")

In [ ]:
before = len(subset)
subset = subset.dropna(subset=["lag_28"]).reset_index(drop=True)
print(f"Dropped {before - len(subset)} rows lacking full lag history")

In [ ]:
print(subset.isna().sum()[subset.isna().sum() > 0])
print(subset.dtypes)
print(subset["date"].min(), subset["date"].max())

In [ ]:
subset.to_parquet("/kaggle/working/retailcast_features.parquet", index=False)
print("Saved: retailcast_features.parquet")
print(f"Final shape: {subset.shape}")